# @zz-windfm-test/building-data-genome-project-2

Hourly meter readings from 1,636 non-residential buildings at 19 sites, with metadata and weather, 2016-2017.

Published version **v1.0.0** — this notebook fetches that exact snapshot from the
PowerAI Hub, not the upstream repository's latest commit.

Run all cells: **Runtime → Run all** (or ⌘/Ctrl + F9).

The Hub does not run primitives. This notebook runs in *your* Colab session, on
Google's hardware, under your account.

[View on the Hub](https://hub.powerai.ai/zz-windfm-test/building-data-genome-project-2)


In [ ]:
PRIMITIVE = "@zz-windfm-test/building-data-genome-project-2"
ARCHIVE = "https://hub-api.powerai.ai/api/artifacts/zz-windfm-test/building-data-genome-project-2/download-archive/"
WORKDIR = "/content/primitive"

import shutil, urllib.request, zipfile
from pathlib import Path

shutil.rmtree(WORKDIR, ignore_errors=True)          # re-runs start clean
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

archive = Path("/content/primitive.zip")
urllib.request.urlretrieve(ARCHIVE, archive)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(WORKDIR)

files = sorted(p for p in Path(WORKDIR).rglob("*") if p.is_file())
print(f"{PRIMITIVE}: {len(files)} files in {WORKDIR}")
for p in files[:20]:
    print(" ", p.relative_to(WORKDIR))
if len(files) > 20:
    print(f"  … and {len(files) - 20} more")


In [ ]:
import importlib.metadata as md, subprocess, sys
from pathlib import Path

print(f"Python {sys.version.split()[0]}")

# The file nearest the root first: a requirements.txt under docs/ or tests/ is not this
# primitive's, and rglob alone would pick whichever it met first.
found = sorted(
    Path(WORKDIR).rglob("requirements.txt"),
    key=lambda p: (len(p.relative_to(WORKDIR).parts), str(p)),
)
if not found:
    print("No requirements.txt. Install what you need with pip, e.g. !pip install pandas")
else:
    req = found[0]
    print(f"Installing {req.relative_to(WORKDIR)} …")
    if len(found) > 1:
        others = ", ".join(str(p.relative_to(WORKDIR)) for p in found[1:4])
        print(f"Not installed (only one file is): {others}")

    def versions():
        return {d.metadata["Name"].lower(): d.version for d in md.distributions() if d.metadata["Name"]}

    before = versions()
    done = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        capture_output=True, text=True,
    )
    if done.returncode == 0:
        print("done")
        after = versions()
        # An install can replace what Colab ships; say so, so a later surprise has a cause.
        for name in sorted(n for n in before if n in after and before[n] != after[n])[:12]:
            print(f"  changed: {name} {before[name]} -> {after[name]}")
    else:
        print(
            "pip failed. pip installs the whole file or none of it, so one package that "
            "cannot be built on this Python stops all of it:"
        )
        print(done.stderr[-2000:])


In [ ]:
import os
os.chdir(WORKDIR)

readme = next((p for p in sorted(Path(WORKDIR).rglob("README*")) if p.is_file()), None)
print(readme.read_text(errors="replace")[:3000] if readme else "No README in this primitive.")


## Your turn

`@zz-windfm-test/building-data-genome-project-2` is unpacked in `/content/primitive` and it is the working directory. Add a
cell and use it.

Nothing here is sent back to the Hub — this session is yours, and it disappears when
you close it. Colab's free tier gives you CPU always and a GPU when one is spare.
